# 📘 CIFAR-10 Image Classification Learning Project

## Build and Compare **ANN vs CNN** on CIFAR-10

This notebook is designed for **students and beginners** to learn:
- How image classification works
- Why **CNN performs better than ANN**
- How architecture impacts performance
- How training strategies improve results

> 🎯 **Learning Goal:** Understand the complete DL pipeline by **reading the markdown + running the ready code**.

## 🧠 Problem Statement

Build an image classification model on the **CIFAR-10 dataset** using:

1. **Artificial Neural Network (ANN)**
2. **Convolutional Neural Network (CNN)**

Then compare: **Accuracy**, **Loss curves**, **Generalization**, **Training strategies**

---
### 📦 CIFAR-10 Classes
Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print('TensorFlow version:', tf.__version__)

## 📥 Load Dataset

We use **CIFAR-10**, which contains **60,000 color images of size 32x32x3**.
- 50,000 training images
- 10,000 test images

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print('Train shape:', x_train.shape)
print('Test shape: ', x_test.shape)

## 🖼️ Visualize Sample Images

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')
plt.suptitle('CIFAR-10 Sample Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## ✏️ Preprocessing

We normalize pixel values from **0-255 to 0-1** so training becomes stable.

In [ ]:
# Normalize to [0, 1]
x_train_norm = x_train / 255.0
x_test_norm  = x_test  / 255.0

# Flatten for ANN: (50000, 32, 32, 3) -> (50000, 3072)
x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat  = x_test_norm.reshape(len(x_test_norm), -1)

print('Flat shape for ANN:', x_train_flat.shape)
print('2D  shape for CNN :', x_train_norm.shape)

## 🔷 Part 1: ANN Model

ANN treats images as **flat vectors**, so it cannot preserve spatial features.  
This helps students understand **why CNN is better for images**.

> ✅ **Student Task 1 completed:** Increased ANN layers — 1024 → 512 → 256 → 10

In [ ]:
ann_model = models.Sequential([
    layers.Dense(1024, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.4),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='ANN_Model')

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

In [ ]:
# Student Task 3: epochs=20  |  Student Task 4: EarlyStopping
early_stop_ann = EarlyStopping(
    monitor='val_accuracy', patience=4,
    restore_best_weights=True, verbose=1
)

ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop_ann],
    verbose=1
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test, verbose=0)
print(f'ANN Test Accuracy : {ann_test_acc:.4f}')
print(f'ANN Test Loss     : {ann_test_loss:.4f}')

## 🔷 Part 2: CNN Model

CNN preserves **spatial relationships** using:
- Convolution layers
- Pooling
- Feature extraction
- Hierarchical learning

This is why CNN performs much better for image tasks.

> ✅ **Student Task 2 completed:** Filter sizes scaled 32 → 64 → 128 (3 conv blocks)

In [ ]:
cnn_model = models.Sequential([
    # Block 1: 32 filters
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 2: 64 filters
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 3: 128 filters
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Classifier Head
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN_Model')

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
early_stop_cnn = EarlyStopping(
    monitor='val_accuracy', patience=4,
    restore_best_weights=True, verbose=1
)

cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop_cnn],
    verbose=1
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f'CNN Test Accuracy : {cnn_test_acc:.4f}')
print(f'CNN Test Loss     : {cnn_test_loss:.4f}')

## 📈 Compare Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(ann_history.history['val_accuracy'], label='ANN Val',   color='#E07B54', lw=2)
axes[0].plot(cnn_history.history['val_accuracy'], label='CNN Val',   color='#4A90D9', lw=2)
axes[0].plot(ann_history.history['accuracy'],     label='ANN Train', color='#E07B54', lw=1.5, linestyle='--')
axes[0].plot(cnn_history.history['accuracy'],     label='CNN Train', color='#4A90D9', lw=1.5, linestyle='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('ANN vs CNN Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(ann_history.history['val_loss'], label='ANN Val Loss', color='#E07B54', lw=2)
axes[1].plot(cnn_history.history['val_loss'], label='CNN Val Loss', color='#4A90D9', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('ANN vs CNN Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Training Strategy Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🚀 Training Strategy Upgrade: Data Augmentation

This strategy improves generalization by generating transformed images.
- **RandomFlip** – horizontally mirrors images
- **RandomRotation** – slight angle rotation
- **RandomZoom** – slight zoom in/out

> ✅ **Student Task 5 completed:** Full augmented CNN training run executed below.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

aug_cnn_model = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='Augmented_CNN')

aug_cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

aug_cnn_model.summary()

In [ ]:
early_stop_aug = EarlyStopping(
    monitor='val_accuracy', patience=5,
    restore_best_weights=True, verbose=1
)

aug_history = aug_cnn_model.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop_aug],
    verbose=1
)

aug_test_loss, aug_test_acc = aug_cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f'Augmented CNN Test Accuracy: {aug_test_acc:.4f}')

## 📊 Final Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Model':         ['ANN', 'CNN', 'Augmented CNN'],
    'Test Accuracy': [ann_test_acc, cnn_test_acc, aug_test_acc],
    'Test Loss':     [ann_test_loss, cnn_test_loss, aug_test_loss],
    'Parameters':    [ann_model.count_params(),
                      cnn_model.count_params(),
                      aug_cnn_model.count_params()]
})

display_df = comparison.copy()
display_df['Test Accuracy'] = display_df['Test Accuracy'].map('{:.2%}'.format)
display_df['Test Loss']     = display_df['Test Loss'].map('{:.4f}'.format)
display_df['Parameters']    = display_df['Parameters'].map('{:,}'.format)
print(display_df.to_string(index=False))
display_df

In [ ]:
model_names = ['ANN', 'CNN', 'Aug CNN']
accs   = [ann_test_acc, cnn_test_acc, aug_test_acc]
colors = ['#E07B54', '#4A90D9', '#44BB77']

plt.figure(figsize=(8, 5))
bars = plt.bar(model_names, [a*100 for a in accs], color=colors, width=0.5, edgecolor='white')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc:.2%}', ha='center', va='bottom', fontweight='bold', fontsize=13)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.title('CIFAR-10: Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylim(0, 105)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(ann_history.history['val_accuracy'], label='ANN',           color='#E07B54', lw=2)
plt.plot(cnn_history.history['val_accuracy'], label='CNN',           color='#4A90D9', lw=2)
plt.plot(aug_history.history['val_accuracy'], label='Augmented CNN', color='#44BB77', lw=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.title('All Models: Validation Accuracy Over Epochs', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🖼️ Sample Predictions (Best Model: CNN)

In [ ]:
np.random.seed(42)
idx = np.random.choice(len(x_test), 10, replace=False)

cnn_preds   = np.argmax(cnn_model.predict(x_test_norm[idx], verbose=0), axis=1)
true_labels = y_test[idx].flatten()

plt.figure(figsize=(14, 4))
for i, (img_i, pred, true) in enumerate(zip(idx, cnn_preds, true_labels)):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_test[img_i])
    color = 'green' if pred == true else 'red'
    plt.title(f'P: {class_names[pred]}\nT: {class_names[true]}',
              color=color, fontsize=8)
    plt.axis('off')
plt.suptitle('CNN Predictions  |  Green=Correct, Red=Wrong',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 🎓 Student Learning Tasks

### Beginner Tasks — ALL COMPLETED

| # | Task | Status |
|---|------|--------|
| 1 | Increase ANN layers | Done: 1024->512->256->10 |
| 2 | Change CNN filters 32->64->128 | Done: 3 conv blocks |
| 3 | Increase epochs to 20 | Done: all models use max 20 epochs |
| 4 | Add EarlyStopping | Done: patience=4 on val_accuracy |
| 5 | Add data augmentation training | Done: full augmented CNN run |

## Conclusion

- **ANN works** but ignores image structure — treats 32x32x3 as flat 3072 numbers
- **CNN extracts spatial features**, so it performs significantly better
- **Training strategies** like dropout, batch norm, and augmentation improve results
- This project builds strong fundamentals for **computer vision interviews and deep learning projects**

### Typical Expected Results on Real CIFAR-10:
| Model | Expected Test Accuracy |
|-------|------------------------|
| ANN (flat) | ~50-55% |
| CNN (3 blocks) | ~70-75% |
| CNN + Augmentation | ~75-80% |